In [1]:
import moondream as md
from PIL import Image

In [2]:
# Initialize with local model path. Can also read .mf.gz files, but we recommend decompressing
# up-front to avoid decompression overhead every time the model is initialized.
model = md.vl(model="moondream-2b-int8.mf")

In [9]:
from nazi_symbols_classification.training.data_preparation import (
    download_data_from_roboflow, reorganise_images, remap_images, get_image_paths
)

In [10]:
image_paths = get_image_paths(path="./datasets/nazi-symbols-classification",
                              sub_folders=("train", "test", "valid"))
len(image_paths)

3370

In [13]:
def classify_document(doc_path, prompts):
    image = Image.open(doc_path)
    encoded_image = model.encode_image(image)

    result_dict = dict()
    for prompt in prompts:
        answer = model.query(encoded_image, prompt)["answer"]
        result_dict[prompt] = ("Yes" in answer)

    return result_dict

In [14]:
prompts = {
    "Analyse whether the image contains a black sun symbol, consisting of concentric circles with radiating, rune-like spokes, associated with Nazi occultism?": "black_sun",
    "Analyse whether the image contains the British Union of Fascists logo, a black lightning bolt set within a white circle on a dark background, symbolizing their fascist ideology.": "british_union_of_fascist",
    "Analyse whether the image contains a broken sun cross symbol, featuring a circle divided into four or more segments by straight lines, often associated with white supremacist or neo-Nazi groups.": "broken_sun_cross",
    "Analyse whether the image is one of the historical images of Adolf Hitler addressing crowds, giving speeches, or leading Nazi rallies during the 1930s and 1940s.": "hitler",
    "Analyse whether the image is one of the images of individuals performing the Hitler salute during historical Nazi Germany events, characterized by a raised right arm held at an angle.": "hitler_salute",
    "Analyse whether the image is one of the images of the Judenstern, the yellow Star of David badge used during the Holocaust, often featuring the word 'Jude' in black lettering in the center.": "judenstern",
    "Analyse whether the image is the imagery of neo-Nazi groups featuring hate symbols like swastikas, Black Sun, Siegrune, or Celtic Cross on flags, banners, clothing, or graffiti, often seen at rallies, protests, or in propaganda materials promoting white supremacy and far-right ideology.": "kolovrat",
    "Analyse whether the image contains a single angular rune shaped like a lightning bolt or elongated 'S,' used in Nazi and neo-Nazi iconography.": "siegrune",
    "Analyse whether the image contains a skull and crossbones insignia, often used by the Nazi SS, with a sinister and militaristic design.": "ss_skull",
    "Analyse whether the image contains an image with the sign of sturmabteilung emblem": "sturmabteilung_emblem",
    "Analyse whether the image contains a black swastika symbol with arms bent at 90 degrees, typically rotated at a 45-degree angle, often shown on a red circular background or a white circle, used during World War II by Nazi Germany.": "swastika",
    "Analyse whether the image contains the Wolfsangel symbol, resembling a hook-like rune, used by Nazi groups and German military units during World War II.": "wolfsangel",
    "Analyse whether the image contains no nazi related content": "non-nazi",
}

In [ ]:
result = dict()

for image_path in image_paths:
    classify_result = classify_document(image_path, prompts.keys())
    result[image_path] = classify_result

In [ ]:
import json


with open("moondream_result_v2.json", "w") as f:
    json.dump(result, f)

In [ ]:
import json


with open("moondream_result_v2.json", "r") as f:
    result = json.load(f)

In [ ]:
import os

classification_dict = dict()
for image_path, classify_result in result.items():
    top5 = sorted(classify_result.items(), key=lambda x: x[1], reverse=True)[:5]
    predicted = [prompts[prompt] for prompt, prob in top5 if prob >= 0.01]
    if not predicted:
        predicted = [prompts[prompt] for prompt, prob in top5[:1]]
    label = os.path.basename(os.path.dirname(image_path))
    classification_dict[image_path] = dict(predicted=predicted, label=label)

classification_dict